---
title: "最小生成树——Kruskal & Prim 算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [7]:
#| code-fold: true
from typing import List
import heapq


class UnionFind:
    def __init__(self, n):
        self.count = n
        self.parent = list(range(n))

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        find_x = self.find(x)
        find_y = self.find(y)
        if find_x != find_y:
            self.parent[find_x] = find_y
            self.count -= 1
            return True
        return False

## 📝 题目 1584：连接所有点的最小费用

::: {.callout-caution}
### 📖 题目描述
给你一个 `points` 数组，表示 2D 平面上的一些点，其中 `points[i] = [xi, yi]` 。

连接点 `[xi, yi]` 和点 `[xj, yj]` 的费用为它们之间的 **曼哈顿距离** ：`|xi - xj| + |yi - yj|` ，其中 `|val|` 表示 `val` 的绝对值。

请你返回将所有点连接起来（任意两点之间都可以建边）的最小总费用。要求连接后，任意两点之间有且仅有一条简单路径（即形成一棵树）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`points = [[0,0],[2,2],[3,10],[5,2],[7,0]]`
    * **输出**：`20`
    * **解释**：我们可以用总费用 20 将所有点连接起来。

* **示例 2**：
    * **输入**：`points = [[3,12],[-2,5],[-4,1]]`
    * **输出**：`18`
:::

In [9]:
class Solution1584:
    def minCostConnectPoints_Kruskal(self, points: List[List[int]]) -> int:
        n = len(points)
        uf = UnionFind(n)
        edges = []
        for i in range(n):  # 收集所有可能的边 (完全图)
            for j in range(i + 1, n):
                dist = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])  # 计算曼哈顿距离
                edges.append((dist, i, j))
        edges.sort()  # 贪心策略：按距离从小到大排序
        res, edges_added = 0, 0  # 遍历边，组装最小生成树
        for dist, i, j in edges:
            if uf.union(i, j):   # 只要 union 成功（没形成环），就采用这条边
                res += dist
                edges_added += 1
                if edges_added == n - 1:  # n 个点只需要 n - 1 条边就能连成树
                    break
        return res

    def minCostConnectPoints_Prim(self, points: List[List[int]]) -> int:
        n = len(points)
        if n <= 1:
            return 0
        adj = {i: [] for i in range(n)}  # 构建邻接表， adj[i] 存放点 i 能连出去的所有边：(距离, 邻居节点)
        for i in range(n):
            for j in range(i + 1, n):
                dist = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])
                adj[i].append((dist, j))
                adj[j].append((dist, i))
        res = 0
        visited = set()
        min_heap = [(0, 0)]  # 最小堆：存放 (距离, 目标点)
        while len(visited) < n:
            dist, curr_node = heapq.heappop(min_heap)
            if curr_node in visited:  # 如果这个点已经在了，说明修这条路会形成环，直接废弃
                continue
            visited.add(curr_node)  # 标记为已使用，并加上费用
            res += dist
            for next_dist, next_node in adj[curr_node]:  # 把这个新节点能连出去的所有路，扔进堆里，作为未来的候选
                if next_node not in visited:  # 优化：只把还没使用的节点的边扔进去
                    heapq.heappush(min_heap, (next_dist, next_node))
        return res

In [10]:
#| code-fold: true
def test_1584(func):
    test_cases = [
        {"desc": "常规示例 1", "points": [[0,0],[2,2],[3,10],[5,2],[7,0]], "expected": 20},
        {"desc": "常规示例 2 (含负数)", "points": [[3,12],[-2,5],[-4,1]], "expected": 18},
        {"desc": "极限: 只有 1 个点", "points": [[0,0]], "expected": 0},
        {"desc": "极限: 只有 2 个点", "points": [[-5,-5],[5,5]], "expected": 20},
        {"desc": "四个点构成正方形", "points": [[0,0],[0,2],[2,0],[2,2]], "expected": 6},
        {"desc": "所有点在同一水平线上", "points": [[0,0],[1,0],[3,0],[6,0]], "expected": 6},
        {"desc": "所有点在同一垂直线上", "points": [[1,-2],[1,0],[1,5],[1,10]], "expected": 12},
        {"desc": "星型辐射 (中心点连所有点最优)", "points": [[0,0],[0,1],[0,-1],[1,0],[-1,0]], "expected": 4},
        {"desc": "存在坐标完全重叠的点", "points": [[1,1],[1,1],[2,2],[2,2]], "expected": 2}, # 0 + 2 + 0 = 2
        {"desc": "大跨度坐标", "points": [[-1000,-1000],[1000,1000]], "expected": 4000},
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["points"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1584(Solution1584().minCostConnectPoints_Prim)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 20   | 20   | 常规示例 1
2    | ✅ PASS | 18   | 18   | 常规示例 2 (含负数)
3    | ✅ PASS | 0    | 0    | 极限: 只有 1 个点
4    | ✅ PASS | 20   | 20   | 极限: 只有 2 个点
5    | ✅ PASS | 6    | 6    | 四个点构成正方形
6    | ✅ PASS | 6    | 6    | 所有点在同一水平线上
7    | ✅ PASS | 12   | 12   | 所有点在同一垂直线上
8    | ✅ PASS | 4    | 4    | 星型辐射 (中心点连所有点最优)
9    | ✅ PASS | 2    | 2    | 存在坐标完全重叠的点
10   | ✅ PASS | 4000 | 4000 | 大跨度坐标
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题没有直接给你“边”，而是给了你“点”。所以我们的第一步是**建图**：算出任意两个点之间的曼哈顿距离。如果有 $N$ 个点，就会有 $\frac{N(N-1)}{2}$ 条边。

#### ⚔️ 玩法一：Kruskal 算法（以“边”为核心，配合并查集）
既然你懂并查集，Kruskal 对你来说只有三步：
1. **收集所有边**：把所有两点之间的边提取出来，存成 `(距离, 点A, 点B)` 的格式。
2. **按距离从小到大排序**：贪心思想，越便宜的边我越先考虑。
3. **并查集连线**：遍历排好序的边，如果 点A 和 点B **不在同一个集合**（即连上不会形成环），就选这条边！把它俩 `union` 起来，费用加上这条边的距离。
   *什么时候结束？* 当你成功连了 `N - 1` 条边时，所有点必定连通，直接提前 `break`！

#### 🛡️ 玩法二：Prim 算法（以“点”为核心，配合最小堆）
Prim 的视角变了，它像是一个不断扩张的“感染区”。
1. **随便挑一个起点**（比如点 0），把它标记为“已在树中”。
2. 把这个点连出去的**所有边** `(距离, 目标点)` 扔进一个**最小堆 (Min-Heap / `heapq`)** 里。
3. 只要堆不为空，就弹出堆顶（当前能摸到的最便宜的边）：
   - 如果这个边指向的“目标点”**已经在树中**了，说明是废边（会形成环），直接 `continue` 丢弃。
   - 如果“目标点”不在树中，太好了！费用加上这段距离，把这个目标点标记为“已在树中”。
   - 接着，把这个新加入的目标点连出去的所有边，也通通扔进最小堆里，继续扩张！
4. 当所有点都被标记为“已在树中”时，扩张结束。
:::

::: {.callout-note}
### 💡 复杂度分析
* **点数 $V$**：`len(points)`，**边数 $E$**：$O(V^2)$（完全图）。
* **Kruskal**：重点在排序，时间复杂度 $O(E \log E)$。适合边少的稀疏图。
* **Prim**：重点在堆操作，时间复杂度 $O(E \log V)$。在这道题（边极多的稠密图）中，Prim 理论上表现更好，但 Python 中 Kruskal 的 `list.sort()` 底层优化极佳，两者速度都很出色。
:::

## 📝 题目 1168：水资源分配优化

::: {.callout-caution}
### 📖 题目描述
村里面一共有 `n` 栋房子，我们希望通过建造水井和铺设管道来为所有房子供水。

对于每个房子 `i`，我们有两种方式为它供水：
1. **自己建水井**：直接在第 `i` 栋房子建水井的成本记录在数组 `wells` 中，其中 `wells[i-1]` 是建井的费用。
2. **铺设管道**：从另一个有水的房子接水管过来。数组 `pipes` 记录了管道信息，其中 `pipes[j] = [house1, house2, cost]` 代表用成本 `cost` 连接 `house1` 和 `house2`。

请你计算并返回为**所有房子**都供水的 **最低总成本**。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3`, `wells = [1, 2, 2]`, `pipes = [[1, 2, 1], [2, 3, 1]]`
    * **输出**：`3`
    * **解释**：
      房子 1 自己建水井，花费 1（此时房子 1 有水了）。
      铺设管道连接房子 1 和 2，花费 1（此时房子 2 有水了）。
      铺设管道连接房子 2 和 3，花费 1（此时房子 3 有水了）。
      总花费：1 + 1 + 1 = 3。

* **示例 2**：
    * **输入**：`n = 2`, `wells = [1, 1]`, `pipes = [[1, 2, 1]]`
    * **输出**：`2`
    * **解释**：
      方案一：房子 1 和 2 各自建水井，花费 1 + 1 = 2。
      方案二：房子 1 建水井花费 1，连接房子 1 和 2 花费 1，总计 1 + 1 = 2。
      最低总成本为 2。

* **示例 3**：
    * **输入**：`n = 2`, `wells = [1, 5]`, `pipes = [[1, 2, 10]]`
    * **输出**：`6`
    * **解释**：
      铺设管道的成本 (10) 远大于房子 2 自己建井的成本 (5)。
      因此，房子 1 和房子 2 各自建水井是最优解，花费 1 + 5 = 6。
:::

::: {.callout-important}
### 💡 思路讲解

这道题最大的难点在于“点”（打井）和“边”（铺管道）都带有费用，无法直接套用 MST 模板。
**图论建模神技：引入“虚拟节点 0”（超级源点）！**
想象存在一个拥有无限水量的“0号水库”。在房子 `i` 花费 `wells[i-1]` 打井，等价于**从“0号水库”修一条花费为 `wells[i-1]` 的管道到房子 `i`**。
这样一来，所有的“打井操作”全部变成了“边”！原问题完美转化为：**在一个包含 `0` 到 `n`（共 `n+1` 个点）的图中，寻找最小生成树。**

#### ⚔️ 玩法一：Kruskal 算法（以“边”为核心）
1. **收集边**：
   - 遍历 `wells`，把每个房子的打井费用变成边 `(wells[i-1], 0, i)` 加入边集。
   - 遍历 `pipes`，把每条管道变成边 `(cost, house1, house2)` 加入边集。
2. **贪心排序**：将所有边按费用从小到大 `sort()`。
3. **并查集连线**：遍历排好序的边，用 `union(u, v)` 判断是否成环。不成环就采用这条边，累加费用。
   - *注意*：现在图里有 `n+1` 个点，所以当连够了 **`n` 条边** 时，整棵树就建成了！

#### 🛡️ 玩法二：Prim 算法（以“点”为核心）
1. **建图 (邻接表)**：
   - 创建包含 `0` 到 `n` 的邻接表 `adj`。
   - 把打井费用作为 `0` 和 `i` 之间的双向边加入 `adj`。
   - 把管道作为 `house1` 和 `house2` 之间的双向边加入 `adj`。
2. **帝国扩张**：
   - 从虚拟节点 `0` 开始扩张（`heappush(min_heap, (0, 0))`），费用为 0。
   - 只要弹出的点不在 `visited` 集合中，就将其占领，累加费用，并把它连出去的所有新边扔进最小堆。
   - 当 `len(visited) == n + 1` 时，所有节点（包含水库和所有房子）全部通水，扩张结束！
:::

::: {.callout-note}
### 💡 复杂度分析
假设房子数量为 $N$，管道数量为 $M$。转化后，总节点数 $V = N + 1$，总边数 $E = N + M$。
* **Kruskal 算法**：
  - **时间复杂度**：$O(E \log E)$。瓶颈在于对所有边进行排序。
  - **空间复杂度**：$O(V + E)$。需要存储边集以及并查集的 `parent` 数组。
* **Prim 算法**：
  - **时间复杂度**：$O(E \log V)$。每条边最多入堆出堆一次。
  - **空间复杂度**：$O(V + E)$。需要存储邻接表以及最小堆。
:::